# Proctoring SDK POC

Tests pre-trained **YOLOv8n**, **YOLOv8s**, **MediaPipe Face Detection**, and **OpenCV quality checks** on all sample images.

**Goal:** Determine if base models can replace LLM-based proctoring checks — no training yet.

---

## 1. Imports & Config

In [ ]:
import cv2
import numpy as np
import mediapipe as mp
import pandas as pd
import json
import os
import urllib.request
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from PIL import Image
from ultralytics import YOLO
from IPython.display import display
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

print("All imports OK")

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
IMAGE_DIR      = Path("images")
ANNOTATED_DIR  = Path("outputs/annotated")
REPORTS_DIR    = Path("outputs/reports")

ANNOTATED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# ── YOLO config ────────────────────────────────────────────────────────────
YOLO_MODELS      = [ "yolo11l.pt","yolo12x.pt"]   # n first, s second
YOLO_IMGSZ       = 640

# Per-class confidence thresholds (COCO class IDs)
# 0=person, 67=cell phone, 73=book, 63=laptop, 62=tv/monitor, 66=keyboard, 64=mouse
PROCTORING_CLASSES = {
    0:  ("person",     0.35),
    67: ("cell_phone", 0.30),
    73: ("book",       0.35),
    63: ("laptop",     0.45),
    62: ("monitor",    0.45),
    66: ("keyboard",   0.35),
    64: ("mouse",      0.35),
}

# ── MediaPipe config ───────────────────────────────────────────────────────
MP_CONFIDENCE    = 0.5   # start here; lower to 0.3 if faces are missed

# ── OpenCV quality thresholds ──────────────────────────────────────────────
BLUR_THRESHOLD        = 55    # Laplacian variance; < 55 = blurry
LOW_LIGHT_THRESHOLD   = 50    # mean brightness; < 50 = dark
OVEREXPOSE_THRESHOLD  = 220   # mean brightness; > 220 = blown out
BLOCKED_BRIGHTNESS    = 20    # very dark
BLOCKED_STD           = 10    # very uniform = camera covered

image_paths = sorted(IMAGE_DIR.glob("*.png")) + sorted(IMAGE_DIR.glob("*.jpg"))
print(f"Found {len(image_paths)} images in {IMAGE_DIR}")

## 2. Helper Functions

In [ ]:
# ── OpenCV quality checks ──────────────────────────────────────────────────

def check_brightness(image_bgr):
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    brightness = float(np.mean(gray))
    if brightness < LOW_LIGHT_THRESHOLD:
        status = "LOW_LIGHT"
    elif brightness > OVEREXPOSE_THRESHOLD:
        status = "OVEREXPOSED"
    else:
        status = "GOOD_LIGHT"
    return status, round(brightness, 2)


def check_blur(image_bgr):
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    blur_score = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    status = "BLURRY" if blur_score < BLUR_THRESHOLD else "SHARP"
    return status, round(blur_score, 2)


def check_blocked(image_bgr):
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    brightness = float(np.mean(gray))
    std_dev    = float(np.std(gray))
    if brightness < BLOCKED_BRIGHTNESS and std_dev < BLOCKED_STD:
        return "CAMERA_BLOCKED"
    return "NOT_BLOCKED"


def run_opencv_checks(image_bgr):
    light_status, brightness = check_brightness(image_bgr)
    blur_status,  blur_score = check_blur(image_bgr)
    blocked_status            = check_blocked(image_bgr)
    return {
        "lighting_status":  light_status,
        "brightness_score": brightness,
        "blur_status":      blur_status,
        "blur_score":       blur_score,
        "blocked_status":   blocked_status,
    }


# ── MediaPipe face detection (Tasks API — mediapipe 0.10+) ─────────────────
# The old mp.solutions.face_detection was removed on Windows in 0.10+.
# We use the new Tasks API with a downloaded .tflite model instead.

_MP_MODEL_PATH = Path("face_detector.tflite")
_MP_MODEL_URL  = (
    "https://storage.googleapis.com/mediapipe-models/"
    "face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite"
)

if not _MP_MODEL_PATH.exists():
    print(f"Downloading MediaPipe face detector model → {_MP_MODEL_PATH} ...")
    urllib.request.urlretrieve(_MP_MODEL_URL, _MP_MODEL_PATH)
    print("  ✓ Model downloaded")
else:
    print(f"  ✓ MediaPipe model already present: {_MP_MODEL_PATH}")

# Build a single reusable detector (cheaper than creating one per image)
_mp_base_opts = mp_python.BaseOptions(model_asset_path=str(_MP_MODEL_PATH))
_mp_opts      = mp_vision.FaceDetectorOptions(
    base_options=_mp_base_opts,
    min_detection_confidence=MP_CONFIDENCE,
)
_mp_detector  = mp_vision.FaceDetector.create_from_options(_mp_opts)


def run_mediapipe(image_bgr):
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    mp_image  = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
    result    = _mp_detector.detect(mp_image)

    faces = result.detections if result.detections else []
    h, w  = image_bgr.shape[:2]
    face_list = []
    for det in faces:
        bb    = det.bounding_box          # absolute pixel coords
        score = det.categories[0].score if det.categories else 0.0
        face_list.append({
            "confidence": round(float(score), 3),
            "bbox": {
                "xmin":   round(bb.origin_x / w, 4),
                "ymin":   round(bb.origin_y / h, 4),
                "width":  round(bb.width    / w, 4),
                "height": round(bb.height   / h, 4),
            }
        })

    return {
        "face_count":     len(face_list),
        "face_present":   len(face_list) > 0,
        "multiple_faces": len(face_list) > 1,
        "faces":          face_list,
    }


# ── Risk flag builder ──────────────────────────────────────────────────────

def compute_risk_flags(yolo_res, mp_res, cv_res):
    flags = []
    detected_labels = {obj["label"] for obj in yolo_res.get("objects", [])}

    if "cell_phone" in detected_labels:
        flags.append("PHONE_DETECTED")
    if yolo_res.get("person_count", 0) > 1:
        flags.append("MULTIPLE_PEOPLE")
    if yolo_res.get("person_count", 0) == 0:
        flags.append("NO_PERSON")
    if "book" in detected_labels:
        flags.append("BOOK_DETECTED")
    if "laptop" in detected_labels:
        flags.append("LAPTOP_DETECTED")
    if "monitor" in detected_labels:
        flags.append("MONITOR_DETECTED")
    if mp_res.get("multiple_faces"):
        flags.append("MULTIPLE_FACES")
    if not mp_res.get("face_present"):
        flags.append("FACE_MISSING")
    if cv_res["lighting_status"] == "LOW_LIGHT":
        flags.append("LOW_LIGHT")
    if cv_res["lighting_status"] == "OVEREXPOSED":
        flags.append("OVEREXPOSED")
    if cv_res["blur_status"] == "BLURRY":
        flags.append("BLURRY")
    if cv_res["blocked_status"] == "CAMERA_BLOCKED":
        flags.append("CAMERA_BLOCKED")
    return flags


print("Helper functions defined")

## 3. Load YOLO Models

In [ ]:
yolo_models = {}
for model_name in YOLO_MODELS:
    print(f"Loading {model_name} ...")
    yolo_models[model_name] = YOLO(model_name)
    print(f"  ✓ {model_name} loaded")

print("\nAll YOLO models ready")

## 4. Run All Checks on Every Image

For each image this cell will:
- Run **YOLOv8n** and **YOLOv8s**
- Run **MediaPipe** face detection
- Run **OpenCV** quality checks
- Save annotated images to `outputs/annotated/`
- Build a combined result record

In [ ]:
def run_yolo_on_image(model, model_name, image_path):
    """Run YOLO and return structured result + annotated image array."""
    results = model(
        str(image_path),
        conf=0.10,          # low global conf; per-class threshold applied below
        imgsz=YOLO_IMGSZ,
        classes=list(PROCTORING_CLASSES.keys()), # ONLY detect these specific classes
        verbose=False,
    )

    objects = []
    annotated_bgr = None
    for result in results:
        annotated_bgr = result.plot()  # numpy BGR array with boxes drawn
        for box in result.boxes:
            cls_id = int(box.cls[0])
            conf   = float(box.conf[0])
            if cls_id not in PROCTORING_CLASSES:
                continue
            label, threshold = PROCTORING_CLASSES[cls_id]
            if conf < threshold:
                continue
            xyxy = box.xyxy[0].tolist()
            objects.append({
                "label":      label,
                "confidence": round(conf, 3),
                "bbox":       [round(v, 1) for v in xyxy],
            })

    person_objs     = [o for o in objects if o["label"] == "person"]
    detected_labels = {o["label"] for o in objects}

    yolo_result = {
        "model":            model_name,
        "objects":          objects,
        "person_count":     len(person_objs),
        "phone_detected":   "cell_phone" in detected_labels,
        "book_detected":    "book"        in detected_labels,
        "laptop_detected":  "laptop"      in detected_labels,
        "monitor_detected": "monitor"     in detected_labels,
    }
    return yolo_result, annotated_bgr


all_results = []

for img_path in image_paths:
    print(f"\n{'─'*60}")
    print(f"Processing: {img_path.name}")

    image_bgr = cv2.imread(str(img_path))
    if image_bgr is None:
        print(f"  ✗ Could not read image, skipping")
        continue

    h, w = image_bgr.shape[:2]

    # ── YOLO ──────────────────────────────────────────────────────────────
    yolo_results_per_model = {}
    for model_name, model in yolo_models.items():
        yolo_res, annotated_bgr = run_yolo_on_image(model, model_name, img_path)
        yolo_results_per_model[model_name] = yolo_res

        stem = img_path.stem
        tag  = model_name.replace(".pt", "").replace("yolov8", "yolo8")
        save_path = ANNOTATED_DIR / f"{stem}_{tag}.jpg"
        if annotated_bgr is not None:
            cv2.imwrite(str(save_path), annotated_bgr)

        labels = [o['label'] for o in yolo_res['objects']]
        print(f"  [{model_name}] detected: {labels if labels else 'nothing'}")

    # ── MediaPipe ─────────────────────────────────────────────────────────
    mp_res = run_mediapipe(image_bgr)
    print(f"  [MediaPipe] faces: {mp_res['face_count']}")

    # ── OpenCV ────────────────────────────────────────────────────────────
    cv_res = run_opencv_checks(image_bgr)
    print(f"  [OpenCV]    light={cv_res['lighting_status']}({cv_res['brightness_score']:.0f})  blur={cv_res['blur_status']}({cv_res['blur_score']:.0f})")

    primary_yolo = yolo_results_per_model.get("yolo12x.pt", yolo_results_per_model.get("yolo11l.pt", list(yolo_results_per_model.values())[0]))
    flags = compute_risk_flags(primary_yolo, mp_res, cv_res)
    if flags:
        print(f"  ⚠  Risk flags: {flags}")

    record = {
        "image":      img_path.name,
        "size":       {"width": w, "height": h},
        "yolo":       yolo_results_per_model,
        "mediapipe":  mp_res,
        "opencv":     cv_res,
        "risk_flags": flags,
    }
    all_results.append(record)

print(f"\n{'='*60}")
print(f"Done. Processed {len(all_results)} images.")

## 5. Save JSON Report

In [ ]:
json_path = REPORTS_DIR / "report.json"
with open(json_path, "w") as f:
    json.dump(all_results, f, indent=2)

print(f"JSON report saved → {json_path}")

## 6. Save CSV Report

In [ ]:
rows = []
for r in all_results:
    yolo_n = r["yolo"].get("yolov8n.pt", {})
    yolo_s = r["yolo"].get("yolov8s.pt", {})
    mp_r   = r["mediapipe"]
    cv_r   = r["opencv"]

    row = {
        "image":               r["image"],
        "width":               r["size"]["width"],
        "height":              r["size"]["height"],

        # YOLOv8n
        "n_person_count":      yolo_n.get("person_count", 0),
        "n_phone":             yolo_n.get("phone_detected", False),
        "n_book":              yolo_n.get("book_detected", False),
        "n_laptop":            yolo_n.get("laptop_detected", False),
        "n_monitor":           yolo_n.get("monitor_detected", False),
        "n_objects_raw":       ", ".join(o["label"] for o in yolo_n.get("objects", [])),

        # YOLOv8s
        "s_person_count":      yolo_s.get("person_count", 0),
        "s_phone":             yolo_s.get("phone_detected", False),
        "s_book":              yolo_s.get("book_detected", False),
        "s_laptop":            yolo_s.get("laptop_detected", False),
        "s_monitor":           yolo_s.get("monitor_detected", False),
        "s_objects_raw":       ", ".join(o["label"] for o in yolo_s.get("objects", [])),

        # MediaPipe
        "face_count":          mp_r.get("face_count", 0),
        "face_present":        mp_r.get("face_present", False),
        "multiple_faces":      mp_r.get("multiple_faces", False),

        # OpenCV
        "lighting_status":     cv_r["lighting_status"],
        "brightness_score":    cv_r["brightness_score"],
        "blur_status":         cv_r["blur_status"],
        "blur_score":          cv_r["blur_score"],
        "blocked_status":      cv_r["blocked_status"],

        # Risk
        "risk_flags":          ", ".join(r["risk_flags"]),
        "risk_count":          len(r["risk_flags"]),
    }
    rows.append(row)

df = pd.DataFrame(rows)
csv_path = REPORTS_DIR / "report.csv"
df.to_csv(csv_path, index=False)

print(f"CSV report saved → {csv_path}")
print(f"Shape: {df.shape}")
df.head()

## 7. Manual Evaluation Table

Review each image result below. Fill the `Expected` and `Correct?` columns manually.

In [ ]:
eval_cols = df[[
    "image",
    "n_person_count", "n_phone", "n_book", "n_laptop",
    "face_count",
    "lighting_status", "blur_status",
    "risk_flags",
]].copy()

eval_cols.columns = [
    "Image",
    "Persons(n)", "Phone(n)", "Book(n)", "Laptop(n)",
    "Faces",
    "Light", "Blur",
    "Risk Flags",
]

eval_cols["Expected"] = ""   # fill manually
eval_cols["Correct?"] = ""   # yes / no / partial
eval_cols["Notes"]    = ""

eval_path = REPORTS_DIR / "evaluation_table.csv"
eval_cols.to_csv(eval_path, index=False)
print(f"Evaluation table saved → {eval_path}")

pd.set_option("display.max_colwidth", 40)
pd.set_option("display.max_rows", 100)
display(eval_cols)

## 8. Summary Statistics

In [ ]:
from collections import Counter

print("═" * 50)
print("SUMMARY STATISTICS")
print("═" * 50)
total = len(df)
print(f"Total images processed : {total}")

print("\n── YOLOv8n ──")
print(f"  Person detected      : {df['n_person_count'].gt(0).sum()} / {total}")
print(f"  Multiple persons     : {df['n_person_count'].gt(1).sum()} / {total}")
print(f"  Phone detected       : {df['n_phone'].sum()} / {total}")
print(f"  Book detected        : {df['n_book'].sum()} / {total}")
print(f"  Laptop detected      : {df['n_laptop'].sum()} / {total}")

print("\n── YOLOv8s ──")
print(f"  Person detected      : {df['s_person_count'].gt(0).sum()} / {total}")
print(f"  Multiple persons     : {df['s_person_count'].gt(1).sum()} / {total}")
print(f"  Phone detected       : {df['s_phone'].sum()} / {total}")
print(f"  Book detected        : {df['s_book'].sum()} / {total}")
print(f"  Laptop detected      : {df['s_laptop'].sum()} / {total}")

print("\n── MediaPipe ──")
print(f"  Face present         : {df['face_present'].sum()} / {total}")
print(f"  Multiple faces       : {df['multiple_faces'].sum()} / {total}")
print(f"  Avg face count       : {df['face_count'].mean():.2f}")

print("\n── OpenCV Quality ──")
print(f"  Good light           : {(df['lighting_status'] == 'GOOD_LIGHT').sum()} / {total}")
print(f"  Low light            : {(df['lighting_status'] == 'LOW_LIGHT').sum()} / {total}")
print(f"  Overexposed          : {(df['lighting_status'] == 'OVEREXPOSED').sum()} / {total}")
print(f"  Sharp                : {(df['blur_status'] == 'SHARP').sum()} / {total}")
print(f"  Blurry               : {(df['blur_status'] == 'BLURRY').sum()} / {total}")
print(f"  Camera blocked       : {(df['blocked_status'] == 'CAMERA_BLOCKED').sum()} / {total}")

print("\n── Risk Flags ──")
all_flags = [f for r in all_results for f in r["risk_flags"]]
for flag, count in Counter(all_flags).most_common():
    print(f"  {flag:<25} : {count}")

## 9. Visualisations

In [ ]:
from collections import Counter

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Proctoring SDK POC — Detection Overview", fontsize=14, fontweight="bold")

# 1. YOLOv8n detections
ax = axes[0, 0]
cat_n = {
    "Person": df['n_person_count'].gt(0).sum(),
    "Phone":  df['n_phone'].sum(),
    "Book":   df['n_book'].sum(),
    "Laptop": df['n_laptop'].sum(),
    "Monitor":df['n_monitor'].sum(),
}
ax.bar(cat_n.keys(), cat_n.values(), color="steelblue", edgecolor="white")
ax.set_title("YOLOv8n — Detections")
ax.set_ylabel("Image count")
ax.set_ylim(0, total + 2)
for i, v in enumerate(cat_n.values()):
    ax.text(i, v + 0.3, str(v), ha="center", fontsize=9)

# 2. YOLOv8s detections
ax = axes[0, 1]
cat_s = {
    "Person": df['s_person_count'].gt(0).sum(),
    "Phone":  df['s_phone'].sum(),
    "Book":   df['s_book'].sum(),
    "Laptop": df['s_laptop'].sum(),
    "Monitor":df['s_monitor'].sum(),
}
ax.bar(cat_s.keys(), cat_s.values(), color="darkorange", edgecolor="white")
ax.set_title("YOLOv8s — Detections")
ax.set_ylabel("Image count")
ax.set_ylim(0, total + 2)
for i, v in enumerate(cat_s.values()):
    ax.text(i, v + 0.3, str(v), ha="center", fontsize=9)

# 3. YOLOv8n vs YOLOv8s comparison
ax = axes[0, 2]
cats  = list(cat_n.keys())
x     = np.arange(len(cats))
width = 0.35
ax.bar(x - width/2, list(cat_n.values()), width, label="YOLOv8n", color="steelblue")
ax.bar(x + width/2, list(cat_s.values()), width, label="YOLOv8s", color="darkorange")
ax.set_title("n vs s — Comparison")
ax.set_xticks(x)
ax.set_xticklabels(cats)
ax.set_ylabel("Image count")
ax.legend()

# 4. MediaPipe face count distribution
ax = axes[1, 0]
face_counts = df['face_count'].value_counts().sort_index()
ax.bar([str(k) for k in face_counts.index], face_counts.values, color="mediumseagreen", edgecolor="white")
ax.set_title("MediaPipe — Face Count Distribution")
ax.set_xlabel("Faces per image")
ax.set_ylabel("Image count")
for i, v in enumerate(face_counts.values):
    ax.text(i, v + 0.2, str(v), ha="center", fontsize=9)

# 5. Lighting status
ax = axes[1, 1]
light_counts = df['lighting_status'].value_counts()
colors_map = {"GOOD_LIGHT": "#4CAF50", "LOW_LIGHT": "#2196F3", "OVEREXPOSED": "#FF9800"}
bar_colors = [colors_map.get(k, "gray") for k in light_counts.index]
ax.bar(light_counts.index, light_counts.values, color=bar_colors, edgecolor="white")
ax.set_title("OpenCV — Lighting Status")
ax.set_ylabel("Image count")
for i, v in enumerate(light_counts.values):
    ax.text(i, v + 0.2, str(v), ha="center", fontsize=9)

# 6. Risk flag frequency
ax = axes[1, 2]
all_flags = [f for r in all_results for f in r["risk_flags"]]
if all_flags:
    flag_counts  = Counter(all_flags)
    sorted_flags = sorted(flag_counts.items(), key=lambda x: x[1], reverse=True)
    flabels, fvals = zip(*sorted_flags)
    bars = ax.barh(flabels, fvals, color="tomato", edgecolor="white")
    ax.set_title("Risk Flag Frequency")
    ax.set_xlabel("Count")
    ax.invert_yaxis()
    for bar, val in zip(bars, fvals):
        ax.text(val + 0.1, bar.get_y() + bar.get_height()/2, str(val), va="center", fontsize=9)
else:
    ax.text(0.5, 0.5, "No risk flags", ha="center", va="center", transform=ax.transAxes)
    ax.set_title("Risk Flag Frequency")

plt.tight_layout()
chart_path = REPORTS_DIR / "overview_chart.png"
plt.savefig(chart_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Chart saved → {chart_path}")

## 10. Spot-Check: View Annotated Images

Visual check of a sample of annotated images with YOLO bounding boxes.

In [ ]:
SHOW_N = 6

annotated_files = sorted(ANNOTATED_DIR.glob("*yolo8n.jpg"))[:SHOW_N]

if not annotated_files:
    print("No annotated images found — run section 4 first")
else:
    cols = 3
    rows = (len(annotated_files) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 5))
    axes = axes.flatten()

    for i, img_file in enumerate(annotated_files):
        img_rgb = cv2.cvtColor(cv2.imread(str(img_file)), cv2.COLOR_BGR2RGB)
        axes[i].imshow(img_rgb)
        axes[i].set_title(img_file.stem, fontsize=9)
        axes[i].axis("off")

    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    plt.suptitle("YOLOv8n — Sample Annotated Images", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

## 11. Observations & Recommendation

> **Fill this section after reviewing the results above.**

| Area | Finding | Action |
|------|---------|--------|
| YOLO — Person | | |
| YOLO — Phone | | |
| YOLO — Book | | |
| YOLO — Laptop | | |
| MediaPipe — Face | | |
| OpenCV — Light | | |
| OpenCV — Blur | | |

### Decision

- [ ] Base models work → move to backend integration
- [ ] Partial failures → tune thresholds / try `imgsz=960` / try YOLOv8m
- [ ] Consistent failures → prepare labeled dataset & fine-tune on Colab GPU